In [2]:
# Notebook setup
%load_ext autoreload
%autoreload 2
import os
import sys
nb_dir = os.path.split(os.getcwd())[0]
if nb_dir not in sys.path:
    sys.path.append(nb_dir)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [3]:
from cerebra_atlas_python import CerebrA, setup_logging

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [4]:
setup_logging("INFO")
cerebra = CerebrA()

### Load sample data (EEG 64 channels 10-10 system)

In [27]:
from mne.io import concatenate_raws, read_raw_edf
from mne.datasets import eegbci

subjects = [1]
runs = [4, 8, 12]
raw_fnames = eegbci.load_data(subjects, runs)
raws = [read_raw_edf(f, preload=True) for f in raw_fnames]
# concatenate runs from subject
raw = concatenate_raws(raws)
# make channel names follow standard conventions
eegbci.standardize(raw)

Extracting EDF parameters from /home/carlos/mne_data/MNE-eegbci-data/files/eegmmidb/1.0.0/S001/S001R04.edf...
EDF file detected
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 19999  =      0.000 ...   124.994 secs...
Extracting EDF parameters from /home/carlos/mne_data/MNE-eegbci-data/files/eegmmidb/1.0.0/S001/S001R08.edf...
EDF file detected
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 19999  =      0.000 ...   124.994 secs...
Extracting EDF parameters from /home/carlos/mne_data/MNE-eegbci-data/files/eegmmidb/1.0.0/S001/S001R12.edf...
EDF file detected
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 19999  =      0.000 ...   124.994 secs...


The EEGs were recorded from 64 electrodes as per the international 10-10 system (excluding electrodes Nz, F9, F10, FT9, FT10, A1, A2, TP9, TP10, P9, and P10), as shown in this figure. The numbers below each electrode name indicate the order in which they appear in the records; note that signals in the records are numbered from 0 to 63, while the numbers in the figure range from 1 to 64.

<div style="display:flex;align-items:center;justify-content:center;padding:25px;flex-direction:column"><img  src="../images/10-10-system-eegbci.png" alt="example3d" width=400px></img</div>
</div>

src: https://archive.physionet.org/pn4/eegmmidb/

### Add montage to cerebra_atlas_python/cerebra_mne/downsampled_montages.py

The montages are derived from MNE's standard montages

If the montage is MNE standard this step is not required

If the montage needs to be downsampled then it should be added to cerebra_mne/downsampled_montages.py

**The name format should be like this:**

* MNE_NAME-downsample-whatever

- for example:

* standard_1020-downsample-my-dataset

In [28]:
cerebra.src_space

Computing src space...
BEM              : <ConductorModel | BEM (3 layers) solver=mne>
Source location file  : dict()
Assuming input in millimeters
Assuming input in MRI coordinates

Positions (in meters) and orientations
31553 sources


<SourceSpaces: [<discrete, n_used=31553>] MRI (surface RAS) coords, ~1.9 MiB>

In [29]:
raw.ch_names

['FC5',
 'FC3',
 'FC1',
 'FCz',
 'FC2',
 'FC4',
 'FC6',
 'C5',
 'C3',
 'C1',
 'Cz',
 'C2',
 'C4',
 'C6',
 'CP5',
 'CP3',
 'CP1',
 'CPz',
 'CP2',
 'CP4',
 'CP6',
 'Fp1',
 'Fpz',
 'Fp2',
 'AF7',
 'AF3',
 'AFz',
 'AF4',
 'AF8',
 'F7',
 'F5',
 'F3',
 'F1',
 'Fz',
 'F2',
 'F4',
 'F6',
 'F8',
 'FT7',
 'FT8',
 'T7',
 'T8',
 'T9',
 'T10',
 'TP7',
 'TP8',
 'P7',
 'P5',
 'P3',
 'P1',
 'Pz',
 'P2',
 'P4',
 'P6',
 'P8',
 'PO7',
 'PO3',
 'POz',
 'PO4',
 'PO8',
 'O1',
 'Oz',
 'O2',
 'Iz']

In [30]:
# Add raw.ch_names to downsampled_montages.py
# "standard-1020-downsample-64-10-10" in this case
# name should be the MNE's original montage name + "-downsample" + label

### Make sure the montage was successfully added

In [31]:
from cerebra_atlas_python.cerebra_mne.mne_montage import MontageMNE

montage = MontageMNE.get_montage("standard_1020-downsample-64-10-10")

In [32]:
montage.plot()
pass

# Corregistration

### Steps:

* Fit ICP
* Manually modify:
  
  tX  rX: 

  tY  rY

  tZ  rZ

* Modify head size to adjust for better fit (modify jupyter cell and relaunch corregistration tool)
* All electrodes should be outside the scalp
* Electrodes distance to scalp should me minimized
* When ready, save head-mri trans to ../cerebra_atlas_python/data/cerebra_data/FreeSurfer/subjects/icbm152 as trans.fif (will be renamed and moved to the corregistration folder once the corregistration UI tool is closed)
<div style="display:flex;align-items:center;justify-content:center;padding:25px;flex-direction:row">

<div style="display:flex;align-items:center;justify-content:center;padding:25px;flex-direction:column">
    <img  src="../images/correg_before.png" alt="example3d" width=400px/>
    <small>Corregistration before manual align</small>
</div>
<div style="display:flex;align-items:center;justify-content:center;padding:25px;flex-direction:column">
    <img  src="../images/correg_after.png" alt="example3d" width=400px/>
    <small>After manual align</small>
</div>

</div>
</div>

In [ ]:
%matplotlib qt
cerebra.montage_name = "biosemi256"
cerebra.head_size = 0.10
cerebra.corregistration(
    montage_name=cerebra.montage_name, head_size=cerebra.head_size
)  # Produce head-mri trans and save to subjects_dir/icbm152/trans.fif

 [INFO] 2025-09-12 10:11:17.454 cerebra_mne - corregistration: Save trans file to /home/carlos/Carlos/cerebra_atlas_python/cerebra_atlas_python/data/cerebra_data/FreeSurfer/subjects/icbm152/trans.fif after aligment
 [INFO] 2025-09-12 10:11:17.455 cerebra_mne - corregistration: Will be automatically renamed to /home/carlos/Carlos/cerebra_atlas_python/cerebra_atlas_python/data/cerebra_data/FreeSurfer/subjects/icbm152/corregistration/biosemi256_0.1_trans.fif


<Info | 8 non-empty values
 bads: []
 ch_names: A1, A2, A3, A4, A5, A6, A7, A8, A9, A10, A11, A12, A13, A14, ...
 chs: 256 EEG
 custom_ref_applied: False
 dig: 259 items (3 Cardinal, 256 EEG)
 highpass: 0.0 Hz
 lowpass: 500.0 Hz
 meas_date: unspecified
 nchan: 256
 projs: []
 sfreq: 1000.0 Hz
>
info_path: /tmp/temp_info.fif
subjects_dir: /home/carlos/Carlos/cerebra_atlas_python/cerebra_atlas_python/data/cerebra_data/FreeSurfer/subjects
subject: icbm152
Using pyvistaqt 3d backend.
For automatic theme detection, "darkdetect" has to be installed! You can install it with `pip install darkdetect`
For automatic theme detection, "darkdetect" has to be installed! You can install it with `pip install darkdetect`
    Triangle neighbors and vertex normals...
Using low resolution head model in /home/carlos/Carlos/cerebra_atlas_python/cerebra_atlas_python/data/cerebra_data/FreeSurfer/subjects/icbm152/bem/outer_skin.surf
    Triangle neighbors and vertex normals...
Using fiducials from: /home/carlos

## Visualize everything is OK

In [10]:
import mne

mne.viz.plot_alignment(
    info=cerebra.info,
    trans=cerebra.head_mri_trans,
    fwd=cerebra.forward,
    dig=True,
    src=cerebra.src_space,
    bem=cerebra.bem,
    show_axes=True,
)

AssertionError: self.trans_path does not exist:/home/carlos/Carlos/cerebra_atlas_python/cerebra_atlas_python/data/cerebra_data/FreeSurfer/subjects/icbm152/corregistration/biosemi256_0.1_trans.fif